# Data 303 B explore

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
RAW = Path("/Users/madshaakonsson/Desktop/7-semester/hmm-modelling/data/raw")
COLS = ["co2", "temperature", "humidity"]


def paths(number: int) -> tuple[Path, Path]:
    """303B file and DTU file for a room number. DTU names a few rooms "Office" instead of "Room"."""
    as_str = f"{number:03d}"
    path_303b = RAW / "DATA303B" / f"climify_office_{as_str}.csv"
    path_dtu = RAW / "dtu" / "timeseries" / f"Room {as_str}.csv"
    if not path_dtu.exists():
        path_dtu = RAW / "dtu" / "timeseries" / f"Office {as_str}.csv"
    return path_303b, path_dtu


def load_303b(path: Path) -> pd.DataFrame:
    """`device_<id>_co2` -> `co2`; UTC `Time` -> tz-naive `datetime` truncated to whole seconds (DTU's format)."""
    df = pd.read_csv(path)
    df = df.rename(columns=lambda c: c.split("_")[-1] if c.startswith("device_") else c)
    df["datetime"] = pd.to_datetime(df["Time"], format="ISO8601", utc=True).dt.tz_convert(None).dt.floor("s")
    return df[["datetime", *COLS]]


def load_dtu(path: Path) -> pd.DataFrame:
    """Sensor readings only: drops the valve rows (`closed`, `motor_position`, ...) where all of COLS are NaN."""
    df = pd.read_csv(path).reindex(columns=["datetime", *COLS])
    df["datetime"] = pd.to_datetime(df["datetime"], format="ISO8601")
    return df.dropna(subset=COLS, how="all")

In [3]:
def align(df_303b: pd.DataFrame, df_dtu: pd.DataFrame) -> pd.DataFrame:
    """Outer-join on datetime over the DTU period. `_merge` says which file each reading came from."""
    in_dtu_period = df_303b["datetime"].between(df_dtu["datetime"].min(), df_dtu["datetime"].max())
    return df_dtu.merge(
        df_303b[in_dtu_period], on="datetime", how="outer", suffixes=("_dtu", "_303b"), indicator=True,
    ).sort_values("datetime").reset_index(drop=True)


def mismatches(merged: pd.DataFrame, col: str) -> pd.DataFrame:
    """Matched rows where `col` differs between the two files (NaN == NaN counts as equal)."""
    both = merged[merged["_merge"] == "both"]
    x, y = both[f"{col}_dtu"], both[f"{col}_303b"]
    return both[~((x == y) | (x.isna() & y.isna()))]


def compare(number: int) -> dict:
    path_303b, path_dtu = paths(number)
    merged = align(load_303b(path_303b), load_dtu(path_dtu))
    counts = merged["_merge"].value_counts()
    row = {
        "room": f"{number:03d}",
        "matched": counts.get("both", 0),
        "dtu_only": counts.get("left_only", 0),
        "303b_only": counts.get("right_only", 0),
    }
    for col in COLS:
        row[f"{col}_mismatches"] = len(mismatches(merged, col))
    return row

## One room in detail

In [4]:
ROOM = 9

path_303b, path_dtu = paths(ROOM)
df_303B, df_DTU = load_303b(path_303b), load_dtu(path_dtu)
merged = align(df_303B, df_DTU)

print(f"303B: {df_303B['datetime'].min()} -> {df_303B['datetime'].max()}  ({len(df_303B)} readings)")
print(f"DTU : {df_DTU['datetime'].min()} -> {df_DTU['datetime'].max()}  ({len(df_DTU)} readings)")
merged["_merge"].value_counts()

303B: 2023-01-01 00:02:46 -> 2026-02-27 03:26:07  (152186 readings)
DTU : 2023-11-01 00:01:53 -> 2024-02-09 13:05:40  (14348 readings)


_merge
both          14334
left_only        14
right_only        0
Name: count, dtype: int64

In [5]:
# Same readings in both files? 0 mismatches means identical values at every shared timestamp.
for col in COLS:
    print(f"{col:12s}: {len(mismatches(merged, col))} mismatches of {(merged['_merge'] == 'both').sum()} matched readings")

co2         : 0 mismatches of 14334 matched readings
temperature : 0 mismatches of 14334 matched readings
humidity    : 0 mismatches of 14334 matched readings


In [6]:
# Readings that exist in only one of the files
merged[merged["_merge"] != "both"]

,datetime,co2_dtu,temperature_dtu,humidity_dtu,co2_303b,temperature_303b,humidity_303b,_merge
10356,2024-01-12 13:01:26,1765.0,22.8,36.33,NaN,NaN,NaN,left_only
10357,2024-01-12 13:11:48,1873.0,22.8,36.72,NaN,NaN,NaN,left_only
10358,2024-01-12 13:21:55,2011.0,22.8,37.11,NaN,NaN,NaN,left_only
10359,2024-01-12 13:32:01,2045.0,22.8,37.50,NaN,NaN,NaN,left_only
10360,2024-01-12 13:42:08,2169.0,22.7,37.89,NaN,NaN,NaN,left_only
10361,2024-01-12 13:42:45,5651.0,421.8,10.55,NaN,NaN,NaN,left_only
10362,2024-01-12 13:52:19,2229.0,22.6,37.89,NaN,NaN,NaN,left_only
10363,2024-01-12 14:02:26,2018.0,22.6,37.50,NaN,NaN,NaN,left_only
10364,2024-01-12 14:03:01,5651.0,421.8,10.55,NaN,NaN,NaN,left_only
10365,2024-01-12 14:12:35,2244.0,22.6,38.28,NaN,NaN,NaN,left_only


## All rooms

In [7]:
ROOM_NUMBERS = [1, 3, 4, 5, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19]

summary = pd.DataFrame([compare(n) for n in ROOM_NUMBERS]).set_index("room")
summary

,matched,dtu_only,303b_only,co2_mismatches,temperature_mismatches,humidity_mismatches
room,,,,,,
001,14277,15,0,0,0,0
003,14316,14,1,0,0,0
004,14256,15,0,0,0,0
005,14312,14,0,0,0,0
007,14322,16,0,0,0,0
008,14309,14,0,0,0,0
009,14334,14,0,0,0,0
010,14289,14,0,0,0,0
012,14332,15,0,0,0,0


In [8]:
mismatch_cols = [f"{c}_mismatches" for c in COLS]
assert (summary[mismatch_cols] == 0).all().all(), summary[summary[mismatch_cols].sum(axis=1) > 0]
print("CO2, temperature and humidity are identical at every shared timestamp in every room.")

CO2, temperature and humidity are identical at every shared timestamp in every room.
